# Intersect Community Data Workflow for Hayward, CA

## Overview
This code works with the National Structures Inventory to run the housing unit allocation (HUA) and the person record file (PREC) workflow.

Functions are provided to obtain and clean data required for the version 2.2.0 Housing Unit Allocation. 

## Required Inputs
Program requires the following inputs:
If using the National Structures Inventory there are no required inputs.

[Census API KEY *REQUIRED*](CENSUS_API_KEY.md) See CENSUS_API_KEY.md file for more details.
    
## Output Description
The output of this workflow is a CSV file with the housing unit inventory allocated to a building inventory using the housing unit allocation model.

The output CSV is designed to be used in the Interdependent Networked Community Resilience Modeling Environment (IN-CORE).

IN-CORE is an open source python package that can be used to model the resilience of a community. To download IN-CORE, see:

https://incore.ncsa.illinois.edu/


## Instructions
Users can run the workflow by executing each block of code in the notebook.

## Description of Program
- program:    ncoda_07jv3_HUA_NSI_Hayward
- task:       Start with NSI building inventory, run housing unit allocation algorithm, and then run person record file algorithm
- See github commits for description of program updates
- Current Version: v3 - 
- 2024-02-20 - Combine code from 07c, 07d, and 07e into one notebook
- 2024-05-22 - removed the drop down menu, did not work consistently
- 2025-02-21 - Test with 2020 HUI
- 2025-03-07 - Update to version 2 to differentiate running 2010 and 2020
- 2025-03-27 - V3 Solved issue for states starting with 0 FIPS code - major modification to code
- 2025-09-17 - Adjust code to run a Monte Carlo Simulation
- 2026-04-16 - Running code to explore missing building data
- 2026-05-23 - v2.1.0 PR 136 change to address point prediction
             - Census API Key NOW REQUIRED
- project:    Interdependent Networked Community Resilience Modeling Environment (IN-CORE), Subtask 5.2 - Social Institutions
- funding:	  NIST Financial Assistance Award Numbers: 70NANB15H044 and 70NANB20H008
- funding:	  Southeast Texas Urban Integrated Field Lab Department of Energy DE-SC0023216 
- author:     Nathanael Rosenheim

## Required Citations:
Rosenheim, Nathanael, Roberto Guidotti, Paolo Gardoni & Walter Gillis Peacock. (2021). Integration of detailed household and housing unit characteristic data with critical infrastructure for post-hazard resilience modeling. _Sustainable and Resilient Infrastructure_. 6(6), 385-401. https://doi.org/10.1080/23789689.2019.1681821

Rosenheim, Nathanael (2021) “Detailed Household and Housing Unit Characteristics: Data and Replication Code.” _DesignSafe-CI_. 
https://doi.org/10.17603/ds2-jwf6-s535.

In [1]:
# To reload submodules need to use this magic command to set autoreload on
%load_ext autoreload
%autoreload 2
from pyncoda.ncoda_00g_community_options import *
from IPython.display import display

### How to set up the Community Dictionary
Please review the python code in the file pyncoda/ncoda_00g_community_options.py

In this file you will find a collection of data dictionaries with various ways to setup the inputs for the Housing Unit Allocation process. 

The basic dictionary includes the name of the community, the county FIPS code, your input building inventory file, and key variables in the building inventory file.

In [2]:
# select a community from this list
# if your community is not in this list, add it to the file ncoda_00g_community_options.py
list_community_options(communities_dictionary)

['Lumberton, NC: IN-CORE Building inventory for Robeson County, NC',
 'Galveston, TX: IN-CORE Building inventory for Galveston County, TX',
 'Galveston, TX: NSI Building inventory for Galveston County, TX',
 'Galveston, TX: IN-CORE Building inventory for Galveston Island, TX',
 'Mayfield, KY: NSI Building inventory for Graves County, KY',
 'Beaumont, TX: NSI Building inventory for Jefferson County, TX',
 'Beaumont, TX: Safayet Building inventory for Jefferson County, TX',
 'Pentwater, MI: NSI Building inventory for Oceana County, MI',
 'Seaside, OR: NSI Building inventory for Clatsop County, OR',
 'Lane County, OR: NSI Building inventory for Lane County, OR',
 'Benton County, OR: NSI Building inventory for Benton County, OR',
 'Southeast Texas Urban Integrated Field Lab: NSI Building inventory for Southeast Texas',
 'Southeast Texas Urban Integrated Field Lab (12 neighbor counties): NSI Building inventory for Southeast Texas',
 'Brazos County, TX: NSI Building inventory for Brazos Coun

In [3]:
community_id_by_name = 'Hayward, CA: NSI Building inventory for Alameda County, CA, 2010'

In [4]:
community_id, focalplace, countyname, countyfips = get_community_id_by_name(community_id_by_name)
communities = {community_id : communities_dictionary[community_id]}

Selected community ID: Hayward_CA_NSI
Hayward, CA is in CALIFORNIA
Focal place: Hayward
Hayward, CA is in Alameda County, CA with FIPS code 06001
Use IN-CORE: False


## Setup Python Environment

In [5]:
import pandas as pd
import geopandas as gpd # For reading in shapefiles
import numpy as np
import sys # For displaying package versions
import os # For managing directories and file paths if drive is mounted
import scooby # Reports Python environment

import contextily as cx # For adding basemap tiles to plot
import matplotlib.pyplot as plt # For plotting and making graphs

In [6]:
# open, read, and execute python program with reusable commands
from pyncoda.ncoda_00d_cleanvarsutils import *
from pyncoda.ncoda_04c_poptableresults import *
from pyncoda.ncoda_07i_process_communities import process_community_workflow

In [7]:
# Generate report of Python environment
base_packages = ['pandas','ipyleaflet','seaborn','contextily']
incore_packages = ['pyincore','pyincore_viz']
check_packages = base_packages + incore_packages
print(scooby.Report(additional=check_packages))


--------------------------------------------------------------------------------
  Date: Mon May 25 11:22:42 2026 Eastern Daylight Time

                OS : Windows (10 10.0.26200 SP0 Multiprocessor Free)
            CPU(s) : 16
           Machine : AMD64
      Architecture : 64bit
               RAM : 31.7 GiB
       Environment : Jupyter

  Python 3.10.14 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:44:50)
  [MSC v.1916 64 bit (AMD64)]

            pandas : 2.2.2
        ipyleaflet : Module not found
           seaborn : 0.13.2
        contextily : 1.6.0
          pyincore : Module not found
      pyincore_viz : Module not found
             numpy : 1.26.4
             scipy : 1.13.1
           IPython : 8.25.0
        matplotlib : 3.8.4
            scooby : 0.10.0

  Intel(R) oneAPI Math Kernel Library Version 2023.1-Product Build 20230303
  for Intel(R) 64 architecture applications
--------------------------------------------------------------------------------


In [8]:
# Check working directory - good practice for relative path access
os.getcwd()

'c:\\Users\\nathanael99\\MyProjects\\GitHub\\intersect-community-data'

## Run Housing Unit Allocation
The following code will produce the following outputs:
1. Housing Unit Inventory
2. Address Point Inventory
3. Housing Unit Allocation

In [9]:
basevintage_options = ['2010','2020']

In [10]:
hua_hui_gdf_dict = {}
base_seed = 9876
iterations = 1
# iterate through basevintage options to run Monte Carlo Simulation
for basevintage in basevintage_options:
    hua_hui_gdf_dict[basevintage] = {}
    for i in range(iterations):
        seed_i = base_seed + i
        print(f"Running iteration {i+1} of {iterations} for basevintage {basevintage} with seed {seed_i}")
        workflow = process_community_workflow(
                    communities,
                    seed = seed_i,
                    version = '2.2.0',
                    version_text = 'v2-2-0',
                    basevintage = basevintage,
                    outputfolder ="OutputData",
                    outputfolders = {},
                    savefiles = True)
        hua_hui_gdf_dict[basevintage][seed_i] = workflow.process_communities()

Running iteration 1 of 1 for basevintage 2010 with seed 9876
Generating Housing Unit Inventory v2-2-0 data for Hayward, CA
Alameda County, CA : county FIPS Code 06001
File already exists, skipping: OutputData/Hayward_CA_NSI/../hui_v2-2-0_Hayward_CA_NSI_2010_rs9876.csv
Checking output for huid
Checking output for blockid
Checking output for bgid
Checking output for tractid
Checking output for FIPScounty
Checking output for numprec
Checking output for ownershp
Checking output for race
Checking output for hispan
Checking output for family
Checking output for vacancy
Checking output for gqtype
Checking output for incomegroup
Checking output for hhinc
Checking output for randincome
Checking output for poverty
Checking huid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
Checking blockid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
    Attempt to fix length of blockid
Checking bgid Data Type
   Current t

## Explore and Validate Housing Unit Allocation


### Look at population characteristics and compare to US Census

In [11]:
focalplace = communities[community_id]['community_name']
print(focalplace, focalplace, countyname, countyfips)

Hayward, CA Hayward, CA Alameda County, CA 06001


In [12]:
hua_gdf = hua_hui_gdf_dict['2010'][9876]

In [13]:
PopResultsTable.pop_results_table(
                  input_df = hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

Tenure Status,1 Owner Occupied (%),2 Renter Occupied (%),Total Population by Households (%)
Race Ethnicity,,,
"1 White alone, Not Hispanic","354,848 (43.4%)","176,870 (27.8%)","531,718 (36.5%)"
"2 Black alone, Not Hispanic","59,667 (7.3%)","111,333 (17.5%)","171,000 (11.8%)"
"3 American Indian and Alaska Native alone, Not Hispanic","1,468 (0.2%)","2,057 (0.3%)","3,525 (0.2%)"
"4 Asian alone, Not Hispanic","240,249 (29.4%)","134,042 (21.0%)","374,291 (25.7%)"
"5 Other Race, Not Hispanic","23,368 (2.9%)","25,320 (4.0%)","48,688 (3.3%)"
"6 Any Race, Hispanic","138,722 (17.0%)","187,253 (29.4%)","325,975 (22.4%)"
Total,"818,322 (100.0%)","636,875 (100.0%)","1,455,197 (100.0%)"


In [14]:
PopResultsTable.pop_results_table(hua_gdf, 
                   who = "Median Household Income", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

Tenure Status,1 Owner Occupied,2 Renter Occupied,Median Household Income
Race Ethnicity,,,
"1 White alone, Not Hispanic","$96,313","$69,092","$85,396"
"2 Black alone, Not Hispanic","$51,097","$39,144","$43,041"
"3 American Indian and Alaska Native alone, Not Hispanic","$66,294","$46,984","$53,527"
"4 Asian alone, Not Hispanic","$104,274","$66,249","$88,095"
"5 Other Race, Not Hispanic","$76,302","$57,312","$64,782"
"6 Any Race, Hispanic","$64,425","$51,340","$56,632"
Total,"$88,275","$56,883","$71,405"


In [15]:
hua_gdf.head()

,huid,blockid,bgid,tractid,FIPScounty,numprec,ownershp,race,hispan,family,...,hhinc,randincome,poverty,BLOCKID10_str,fd_id_bid,placeNAME10,huestimate,x,y,geometry
0,B060014001001007H0001,060014001001007,060014001001,06001400100,06001,1,1.0,1.0,0.0,0.0,...,5.0,122331.0,0.0,B060014001001007,nsi-474648750-849VVQ65+J2C-2-2-3-3,Oakland,2.0,-122.242497,37.861551,POINT (-122.24 37.862)
1,B060014001001007H0002,060014001001007,060014001001,06001400100,06001,1,1.0,1.0,0.0,0.0,...,5.0,249702.0,0.0,B060014001001007,nsi-474648764-849VVQ65+PW4-1-3-2-2,Oakland,2.0,-122.240157,37.861752,POINT (-122.24 37.862)
2,B060014001001007H0003,060014001001007,060014001001,06001400100,06001,1,1.0,1.0,0.0,0.0,...,4.0,81115.0,0.0,B060014001001007,nsi-474648761-849VVQ64+JV4-5-4-3-3,Oakland,2.0,-122.242793,37.861523,POINT (-122.24 37.862)
3,B060014001001007H0004,060014001001007,060014001001,06001400100,06001,1,1.0,1.0,0.0,0.0,...,5.0,185826.0,0.0,B060014001001007,nsi-474648762-849VVQ65+MC5-2-3-2-3,Oakland,2.0,-122.241380,37.861648,POINT (-122.24 37.862)
4,B060014001001007H0005,060014001001007,060014001001,06001400100,06001,1,1.0,1.0,0.0,0.0,...,5.0,173579.0,0.0,B060014001001007,nsi-474648765-849VVQ64+PQ7-2-2-2-3,Oakland,2.0,-122.243089,37.861795,POINT (-122.24 37.862)


In [16]:
# Show population counts by placeNAME10 - weight by numprec
hua_gdf.groupby('placeNAME10').size()

placeNAME10
Alameda            32372
Albany              7898
Ashland             7769
Berkeley           49583
Castro Valley      23428
Cherryland          4998
Dublin             15800
Emeryville          6649
Fairview            3702
Fremont            74046
Hayward            48347
Livermore          30396
Milpitas              32
Newark             13432
Oakland           169958
Piedmont            3925
Pleasanton         25956
San Leandro        32459
San Lorenzo         7685
Sunol                394
Unincorporated      3322
Union City         21321
dtype: int64

In [17]:
focalplace

'Hayward, CA'

In [18]:
# set dataframe for focal place
focalplace_hua_gdf =  hua_gdf.loc[hua_gdf['placeNAME10'] == 'Hayward'].copy(deep=True)

In [19]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

Tenure Status,1 Owner Occupied (%),2 Renter Occupied (%),Total Population by Households (%)
Race Ethnicity,,,
"1 White alone, Not Hispanic","18,244 (24.9%)","9,813 (15.1%)","28,057 (20.3%)"
"2 Black alone, Not Hispanic","4,535 (6.2%)","10,175 (15.6%)","14,710 (10.6%)"
"3 American Indian and Alaska Native alone, Not Hispanic",173 (0.2%),224 (0.3%),397 (0.3%)
"4 Asian alone, Not Hispanic","20,695 (28.2%)","8,516 (13.1%)","29,211 (21.1%)"
"5 Other Race, Not Hispanic","3,917 (5.3%)","3,459 (5.3%)","7,376 (5.3%)"
"6 Any Race, Hispanic","25,702 (35.1%)","32,916 (50.6%)","58,618 (42.4%)"
Total,"73,266 (100.0%)","65,103 (100.0%)","138,369 (100.0%)"


In [20]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                   who = "Median Household Income", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

Tenure Status,1 Owner Occupied,2 Renter Occupied,Median Household Income
Race Ethnicity,,,
"1 White alone, Not Hispanic","$67,776","$53,986","$62,442"
"2 Black alone, Not Hispanic","$66,180","$51,543","$55,242"
"3 American Indian and Alaska Native alone, Not Hispanic","$71,456","$44,709","$62,751"
"4 Asian alone, Not Hispanic","$92,130","$70,579","$85,200"
"5 Other Race, Not Hispanic","$72,086","$58,260","$65,074"
"6 Any Race, Hispanic","$61,950","$52,269","$56,488"
Total,"$70,232","$55,004","$62,753"


In [21]:
hua_gdf['fd_id_bid'].describe()

count                                 583472
unique                                389001
top       nsi-475653265-849VRMQW+JPH-0-0-0-0
freq                                     673
Name: fd_id_bid, dtype: object

In [22]:
bldg_uniqueid = 'fd_id_bid'
# add category for missing building id
buildingdata_conditions = {'cat_var' : {'variable_label' : 'Building Data Availability',
                         'notes' : 'Does Housing Unit have building data?'},
              'condition_list' : {
                1 : {'condition': f"(df['{bldg_uniqueid}'] == 'missing building id')", 'value_label': "0 Missing Building Data"},
                2 : {'condition': f"(df['{bldg_uniqueid}'] != 'missing building id')", 'value_label': "1 Building Data Available"}}
            }
hua_gdf = add_label_cat_conditions_df(hua_gdf, conditions = buildingdata_conditions)

0 Missing Building Data had 370 observations
1 Building Data Available had 583102 observations


In [23]:
PopResultsTable.pop_results_table(hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

Building Data Availability_str,0 Missing Building Data (%),1 Building Data Available (%),Total Population by Households (%),Percent Row 0 Missing Building Data
Race Ethnicity,,,,
"1 White alone, Not Hispanic",247 (14.0%),"531,471 (35.6%)","531,718 (35.6%)",0.0%
"2 Black alone, Not Hispanic",125 (7.1%),"170,875 (11.5%)","171,000 (11.5%)",0.1%
"3 American Indian and Alaska Native alone, Not Hispanic",2 (0.1%),"3,523 (0.2%)","3,525 (0.2%)",0.1%
"4 Asian alone, Not Hispanic",235 (13.3%),"374,056 (25.1%)","374,291 (25.1%)",0.1%
"5 Other Race, Not Hispanic",48 (2.7%),"48,640 (3.3%)","48,688 (3.3%)",0.1%
"6 Any Race, Hispanic",205 (11.6%),"325,770 (21.9%)","325,975 (21.8%)",0.1%
7 Group Quarters no Race Ethnicity Data,903 (51.2%),"36,539 (2.5%)","37,442 (2.5%)",2.4%
Total,"1,765 (100.0%)","1,490,874 (100.0%)","1,492,639 (100.0%)",0.1%


In [24]:
focalplace_hua_gdf = add_label_cat_conditions_df(focalplace_hua_gdf, conditions = buildingdata_conditions)

PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

0 Missing Building Data had 8 observations
1 Building Data Available had 48339 observations


Building Data Availability_str,0 Missing Building Data (%),1 Building Data Available (%),Total Population by Households (%),Percent Row 0 Missing Building Data
Race Ethnicity,,,,
"1 White alone, Not Hispanic",nan (nan%),"28,057 (19.9%)","28,057 (19.9%)",nan%
"2 Black alone, Not Hispanic",nan (nan%),"14,710 (10.4%)","14,710 (10.4%)",nan%
"3 American Indian and Alaska Native alone, Not Hispanic",nan (nan%),397 (0.3%),397 (0.3%),nan%
"4 Asian alone, Not Hispanic",6 (15.8%),"29,205 (20.7%)","29,211 (20.7%)",0.0%
"5 Other Race, Not Hispanic",7 (18.4%),"7,369 (5.2%)","7,376 (5.2%)",0.1%
"6 Any Race, Hispanic",25 (65.8%),"58,593 (41.5%)","58,618 (41.5%)",0.0%
7 Group Quarters no Race Ethnicity Data,nan (nan%),"2,724 (1.9%)","2,724 (1.9%)",nan%
Total,38 (100.0%),"141,055 (100.0%)","141,093 (100.0%)",0.0%


In [25]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Tenure Status",
                  where = focalplace,
                  when = "2010",
                  row_index = "Tenure Status",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

Building Data Availability_str,0 Missing Building Data (%),1 Building Data Available (%),Total Population by Households (%),Percent Row 0 Missing Building Data
Tenure Status,,,,
1 Owner Occupied,30 (78.9%),"73,236 (52.9%)","73,266 (52.9%)",0.0%
2 Renter Occupied,8 (21.1%),"65,095 (47.1%)","65,103 (47.1%)",0.0%
Total,38 (100.0%),"138,331 (100.0%)","138,369 (100.0%)",0.0%


In [26]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Income Groups",
                  where = focalplace,
                  when = "2010",
                  row_index = "Household Income Group",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

Building Data Availability_str,0 Missing Building Data (%),1 Building Data Available (%),Total Population by Households (%),Percent Row 0 Missing Building Data
Household Income Group,,,,
"1 Less than $15,000",6 (15.8%),"11,143 (8.1%)","11,149 (8.1%)",0.1%
"2 $15,000 to $24,999",7 (18.4%),"11,089 (8.0%)","11,096 (8.0%)",0.1%
"3 $25,000 to $74,999",20 (52.6%),"58,628 (42.4%)","58,648 (42.4%)",0.0%
"4 $75,000 to $99,999",5 (13.2%),"18,081 (13.1%)","18,086 (13.1%)",0.0%
"5 $100,000 or more",nan (nan%),"39,390 (28.5%)","39,390 (28.5%)",nan%
Total,38 (100.0%),"138,331 (100.0%)","138,369 (100.0%)",0.0%


#### Validate the Housing Unit Allocation has worked
Notice that the population count totals for the community
should match (pretty closely) data collected for the 2010 Decennial Census.
This can be confirmed by going to data.census.gov

In [27]:
print("Total Population by Race and Ethnicity:")
print(f"https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=DECENNIALSF12010.P5")

print("Median Income by Race and Ethnicity:")
print(f"All Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013")
print(f"Black Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013B")
print(f"White, not Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013H")
print(f"Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013I")

Total Population by Race and Ethnicity:
https://data.census.gov/cedsci/table?g=050XX00US06001&tid=DECENNIALSF12010.P5
Median Income by Race and Ethnicity:
All Households: https://data.census.gov/cedsci/table?g=050XX00US06001&tid=ACSDT5Y2012.B19013
Black Households: https://data.census.gov/cedsci/table?g=050XX00US06001&tid=ACSDT5Y2012.B19013B
White, not Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US06001&tid=ACSDT5Y2012.B19013H
Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US06001&tid=ACSDT5Y2012.B19013I


Differences in the housing unit allocation and the Census count may be due to differences between political boundaries and the building inventory. See Rosenheim et al 2019 for more details.

The housing unit allocation, plus the building results will become the input for the social science models such as the population dislocation model.